# チュートリアル 02: マルチターン会話 - ステートフルな旅行アシスタント
## Azure OpenAI Service 版
##  学習目標
このノートブックを終えると、以下ができるようになります:
- セッションとは何か、なぜそれが必須なのかを理解する
- コンテキストを記憶するステートフルな会話を作成する
- Azure AI で会話セッションを管理する
- 旅行アシスタントとの自然で流れるような対話を構築する
- 新しいセッションを作成する場合と既存のセッションを再利用する場合を学ぶ

##  主要な概念

### セッションとは?
**セッション (AgentSession)**は、複数のクエリにわたってコンテキストを維持する会話履歴です。

**セッションなしの場合:**
```
ユーザー: "日本に行きたい"
エージェント: "素晴らしい! 日本には多くのアトラクションがあります..."

ユーザー: "そこの天気はどう?"
エージェント: "どこの天気情報をお知りになりたいですか?"  日本を忘れた!
```

**セッションありの場合:**
```
ユーザー: "日本に行きたい"
エージェント: "素晴らしい! 日本には多くのアトラクションがあります..."

ユーザー: "そこの天気はどう?"
エージェント: "日本の天気を確認させていただきます..."  覚えている!
```

### セッションのタイプ

1. **自動セッション** (ステートレス)
   - `run()` 呼び出しごとに新しいセッションが作成される
   - クエリ間でメモリなし
   - 適している用途: 一回限りの質問

2. **永続的セッション** (ステートフル)
   - 複数の呼び出しで同じセッションが再利用される
   - 完全な会話履歴を維持
   - 適している用途: マルチターン会話

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
import os
from random import randint, choice
from typing import Annotated

from agent_framework import Agent, AgentSession
from pydantic import Field
from dotenv import load_dotenv

load_dotenv(override=True)
print("✅ インポート成功！")

In [ ]:
import os
from agent_framework.azure import AzureOpenAIChatClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: Agent を `async with` で使うため、client も各実行で新しく生成します。
def create_azure_openai_chat_client():
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment
    )

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# 環境変数ベースで設定（Agent Framework は標準の OTEL_* を読む）
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
# Jaeger はトレースのみ対応のため、全シグナル共通の ENDPOINT ではなく
# トレース専用の TRACES_ENDPOINT を設定してメトリクス/ログの送信エラーを防ぐ
os.environ.setdefault("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")

# enable_sensitive_data=True を指定して機微データ(OpenAI の IN/OUT データ)の収集を有効化
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## ステップ 2: 旅行ツールの定義

エージェントが会話全体で使用するツールを作成しましょう。

In [ ]:
def get_weather(
    location: Annotated[str, Field(description="都市名または国名")],
) -> str:
    """指定された場所の現在の天気を返します。"""
    conditions = ["晴れ", "晴れ時々曇り", "曇り", "雨", "風が強い"]
    temp = randint(15, 32)
    condition = choice(conditions)
    return f"{location}の天気: {condition}, {temp}°C"


def get_attractions(
    location: Annotated[str, Field(description="都市名または国名")],
) -> str:
    """目的地の主な観光スポットを返します。"""
    # 観光スポットの擬似データ
    attractions = {
        "Japan": "東京タワー、富士山、京都の寺院、大阪城",
        "Paris": "エッフェル塔、ルーブル美術館、ノートルダム大聖堂、凱旋門",
        "London": "ビッグ・ベン、ロンドン塔、大英博物館、バッキンガム宮殿",
        "Barcelona": "サグラダ・ファミリア、グエル公園、ラ・ランブラ、ゴシック地区",
    }

    for city, attrs in attractions.items():
        if city.lower() in location.lower():
            return f"{location}の主な観光スポット: {attrs}"

    return f"{location}の人気観光スポット: 史跡、美術館、ローカルマーケット、公園"


def estimate_budget(
    destination: Annotated[str, Field(description="目的地の都市または国")],
    days: Annotated[int, Field(description="日数")],
) -> str:
    """目的地と日数から旅行予算を概算します。"""
    daily_costs = {
        "Japan": 150,
        "Paris": 180,
        "London": 200,
        "Barcelona": 120,
        "Thailand": 60,
    }

    cost_per_day = daily_costs.get(destination, 100)
    total = cost_per_day * days

    return f"{destination}で{days}日間の推定予算: ${total}（1日あたり ${cost_per_day}）"


print("✅ ツール定義完了")

## ステップ 3: 問題 - メモリなし (ステートレス)

スレッドなしで何が起こるか見てみましょう - 各クエリは独立しています。

In [ ]:
async def example_without_threads():
    """
    問題点を示します: エージェントが以前の文脈を覚えていません。
    run() のたびに自動的に新しいセッションが作られます。
    """
    print("=== セッションなし (メモリなし) ===")
    print("各クエリは独立 - エージェントは前の会話を忘れます\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="StatelessTravelAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # 1つ目のクエリ
        query1 = "来月日本への旅行を計画しています。"
        print(f"ユーザー: {query1}")
        response1 = await agent.run(query1)
        print(f"エージェント: {response1.text}\n")

        # 2つ目のクエリ - エージェントが日本を覚えている想定
        query2 = "そこの天気はどうですか？"
        print(f"ユーザー: {query2}")
        response2 = await agent.run(query2)
        print(f"エージェント: {response2.text}\n")

        print("❌ 注意: エージェントは「そこ」がどこか知りません！")
        print("   各 run() が別々のセッションを作成しました。\n")


await example_without_threads()

## ステップ 4: 解決策 - 永続的スレッド

では、**永続的スレッド**を使用して会話コンテキストを維持しましょう。

In [ ]:
async def example_with_session():
    """
    This shows the solution: using the same session maintains context.
    """
    print("=== 永続的セッションあり (メモリあり) ===")
    print("同じセッションが複数のクエリ間でコンテキストを維持\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="StatefulTravelAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        # Create a new session that we'll reuse
        session = agent.create_session()

        # First query
        query1 = "来月京都への旅行を計画しています。"
        print(f"ユーザー: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"エージェント: {response1.text}\n")

        # Second query - agent remembers Japan!
        query2 = "そこの天気はどうですか？"
        print(f"ユーザー: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"エージェント: {response2.text}\n")

        # Third query - agent remembers the whole conversation
        query3 = "訪れるべきトップアトラクションは何ですか？"
        print(f"ユーザー: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"エージェント: {response3.text}\n")

        print("✅ 成功: エージェントは会話全体を記憶しています！")
        print(f"   セッションID: {session.session_id}\n")

await example_with_session()

## ステップ 5: 自然なマルチターン計画会話

複数の対話のやり取りがある現実的な旅行計画の会話を行いましょう。

In [ ]:
async def planning_conversation():
    """
    A natural conversation where the agent helps plan a trip.
    """
    print("=== 自然な旅行計画会話 ===")
    print("コンテキストを認識したマルチターン対話\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="TravelPlanningAgent",
            instructions="""
            あなたは旅行計画の専門家です。以下の方法でユーザーの旅行計画を支援してください：
            - 必要に応じて明確化のための質問をする
            - ユーザーが共有するすべての詳細を記憶する
            - ツールを使用して正確な情報を提供する
            - 熱心で親切に対応する
            """,
            tools=[get_weather, get_attractions, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()

        # Conversation flow
        queries = [
            "12月に2週間休みがあり、暖かい場所に行きたいです。",
            "タイかバルセロナを考えています。どちらがおすすめですか？",
            "バルセロナに決めました！12月の天気はどうですか？",
            "7日間でどのくらい予算を見ればいいですか？",
            "それで大丈夫です！必見のアトラクションは何ですか？",
        ]

        for i, query in enumerate(queries, 1):
            print(f"{'='*60}")
            print(f"ターン {i}")
            print(f"{'='*60}")
            print(f"ユーザー: {query}\n")

            response = await agent.run(query, session=session)
            print(f"エージェント: {response.text}\n")

        print("✅ 完全なコンテキストを維持した会話が完了！")
        print(f"   セッションID: {session.session_id}\n")

await planning_conversation()

## ステップ 6: 異なるスレッドでの複数の会話

異なるスレッドを使用して、複数の会話を同時に管理できます。

In [ ]:
async def multiple_conversations():
    """
    Manage multiple separate conversations with different sessions.
    """
    print("=== 複数の会話 (異なるセッション) ===")
    print("各セッションは独自のコンテキストを維持\n")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="MultiConversationAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, get_attractions],
        ) as agent,
    ):
        # Create two separate sessions for two different users/conversations
        alice_session = agent.create_session()
        bob_session = agent.create_session()

        # Alice's conversation about Paris
        print("🧑 Aliceの会話:")
        alice_q1 = "パリを訪れることに興味があります。"
        print(f"Alice: {alice_q1}")
        response = await agent.run(alice_q1, session=alice_session)
        print(f"エージェント: {response.text[:100]}...\n")

        # Bob's conversation about London
        print("👨 Bobの会話:")
        bob_q1 = "次の旅行でロンドンを考えています。"
        print(f"Bob: {bob_q1}")
        response = await agent.run(bob_q1, session=bob_session)
        print(f"エージェント: {response.text[:100]}...\n")

        # Continue Alice's conversation
        print("🧑 Aliceが続けます:")
        alice_q2 = "そこの天気はどうですか？"
        print(f"Alice: {alice_q2}")
        response = await agent.run(alice_q2, session=alice_session)
        print(f"エージェント: {response.text}\n")

        # Continue Bob's conversation
        print("👨 Bobが続けます:")
        bob_q2 = "そこではどのアトラクションを訪れるべきですか？"
        print(f"Bob: {bob_q2}")
        response = await agent.run(bob_q2, session=bob_session)
        print(f"エージェント: {response.text}\n")

        print("✅ 両方の会話が個別のコンテキストを維持！")
        print(f"   AliceのセッションID: {alice_session.session_id}")
        print(f"   BobのセッションID: {bob_session.session_id}\n")

await multiple_conversations()

## ステップ 7: 会話の再開（2つのやり方）

「あとで会話を再開」する方法は、使っている **chat_client がスレッドをサービス側で管理できるか** で変わります。

### (A) Foundry (Azure AI) のサービス管理スレッドで再開
- `thread.service_thread_id` は **サービス側（Foundry）に保存された会話スレッド**を指す ID です。
- この ID を DB 等に保存しておけば、後で `AgentThread(service_thread_id=...)` として再開できます。
- `AzureAIAgentClient(..., thread_id=...)` の `thread_id` は「このクライアントがデフォルトで使うスレッドID（= conversation/thread ID）」という意味です。

### (B) AzureOpenAIChatClient のローカル履歴で再開
- Chat Completions ベースの `AzureOpenAIChatClient` は、通常 **Foundry の thread_id のようなサービス管理スレッド**を前提にしません。
- その場合は `AgentThread.serialize()` の結果（ローカルの会話履歴）を保存し、後で `agent.deserialize_thread(...)` で復元して再開します。

次のセルで (A)/(B) をそれぞれデモします。

In [ ]:
async def resume_conversation_foundry():
    """
    create_azure_openai_chat_client を使ってローカル履歴で会話を再開する例。
    """
    import json

    print("=== AzureOpenAIChatClient: ローカル履歴で会話を再開 ===")

    if not (azure_openai_key and azure_openai_endpoint and api_version and azure_deployment):
        print("⚠️ Azure OpenAI 用の環境変数が不足しています。")
        print("   AZURE_OPENAI_API_KEY / AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_API_VERSION / AZURE_OPENAI_CHAT_DEPLOYMENT")
        return

    saved_session_state: dict | None = None

    print("\n--- パート1: 会話を開始し、session state を保存 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()
        query = "バルセロナに5日間訪れたいです。予算はどのくらいですか？"
        print(f"ユーザー: {query}")
        response = await agent.run(query, session=session)
        print(f"エージェント: {response.text}\n")

        saved_session_state = session.to_dict()
        preview = json.dumps(saved_session_state, ensure_ascii=False)
        print("✅ session state を保存（例: JSON の一部を表示）")
        print(preview[:300] + ("..." if len(preview) > 300 else ""))

    if not saved_session_state:
        print("⚠️ session state の保存に失敗しました。\n")
        return

    print("--- パート2: 保存した session state から会話を再開 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = AgentSession.from_dict(saved_session_state)
        query = "そこの天気はどうですか？"
        print(f"ユーザー: {query}")
        response = await agent.run(query, session=session)
        print(f"エージェント: {response.text}\n")

        print("✅ ローカル履歴から会話を正常に再開！")

await resume_conversation_foundry()

In [ ]:
async def resume_conversation_azure_openai_local():
    """
    (B) AzureOpenAIChatClient のような Chat Completions で「ローカルに会話履歴を保持して」再開する例。
    - ここで保存するのは service_session_id ではなく、AgentSession の serialized state（会話履歴）です。
    - 実アプリでは、この state を DB/Blob/Redis 等に保存します。
    """
    import json

    print("=== (B) AzureOpenAIChatClient: ローカル履歴で会話を再開 ===")

    saved_session_state: dict | None = None

    # Part 1: Start a conversation and save the serialized session state (local history)
    print("\n--- パート1: 会話を開始し、session state を保存 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = agent.create_session()
        query = "バルセロナに5日間訪れたいです。予算はどのくらいですか？"
        print(f"ユーザー: {query}")
        response = await agent.run(query, session=session)
        print(f"エージェント: {response.text}\n")

        saved_session_state = session.to_dict()
        preview = json.dumps(saved_session_state, ensure_ascii=False)

        with open("saved_session_state.json", "w", encoding="utf-8") as f:
            json.dump(saved_session_state, f, ensure_ascii=False, indent=2)

        print("✅ session state を保存（例: JSON の一部を表示）")
        print(preview[:300] + ("..." if len(preview) > 300 else ""))
        print("   (実際のアプリでは、この JSON をデータベース等に保存します)\n")

    if not saved_session_state:
        print("⚠️ session state の保存に失敗しました。\n")
        return

    # Part 2: Resume later by deserializing the saved session state
    print("--- パート2: 保存した session state から会話を再開 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="ResumableLocalAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather, estimate_budget],
        ) as agent,
    ):
        session = AgentSession.from_dict(saved_session_state)
        query = "そこの天気はどうですか？"
        print(f"ユーザー: {query}")
        response = await agent.run(query, session=session)
        print(f"エージェント: {response.text}\n")

        print("✅ ローカル履歴から会話を正常に再開！")
        print("   エージェントは前のセッションからバルセロナを記憶していました。\n")

await resume_conversation_azure_openai_local()

## ステップ 8: スレッドメッセージの検査

スレッドに保存されている会話履歴を調べることができます。

In [ ]:
async def inspect_session():
    """
    セッション内のメッセージと履歴を確認します。
    注: AzureOpenAIChatClient ではセッションはローカル履歴として扱われます。
    """
    print("=== セッションメッセージの検査 ===")

    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="InspectableAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            tools=[get_weather],
        ) as agent,
    ):
        session = agent.create_session()

        # Have a short conversation
        queries = ["東京の天気はどうですか？", "ロンドンはどうですか？", "どちらの天気が良いですか？"]

        print("会話中...\n")
        for i, query in enumerate(queries, 1):
            print(f"クエリ {i}: {query}")
            response = await agent.run(query, session=session)
            print(f"応答: {response.text[:100]}...\n")

        # Inspect the session
        print("=" * 60)
        print(f"セッションID: {session.session_id}")
        print(f"サービス管理セッション: {session.service_session_id is not None}")
        print(f"セッション状態: {session.state}")
        print("=" * 60)
        print("\n注: この構成では、会話履歴は session state としてシリアライズして保存できます。")
        print("必要に応じて AgentSession.from_dict(...) で再開してください。")

await inspect_session()

## 🗑 Microsoft Foundry でのエージェントライフサイクル管理

### エージェントの自動クリーンアップ

**重要:** Microsoft Foundry では、SDK からエージェントを作成する方法によって、実行完了後に**エージェントが自動的に削除されます**

#### 動作の仕組み

```python
async with (
    AzureCliCredential() as credential,
    Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="あなたは旅行アシスタントです。",
    ) as agent,  # ← エージェントがここで作成される
):
    # このブロック内で Azure にエージェントが存在する
    response = await agent.run("こんにちは")
    
# ← Azure からエージェントが削除される (コンテキスト終了時)
```

**何が起こるか:**
1. **エージェント作成** - `async with` ブロックに入ると、Microsoft Foundry でエージェントが作成される
2. **エージェント使用** - クエリの実行、セッションの作成、ツールの使用が可能
3. **エージェント削除** - ブロックを終了すると、Azure からエージェント定義が削除される
4. **セッションは永続化** - セッションデータは Azure に残る (エージェント定義のみが削除される)

### 削除されるものと永続化されるもの

| リソース | コンテキスト終了後 | 理由 |
|----------|-------------------|-----|
| **エージェント定義** |  削除 | 混乱を減らし、コストを節約 |
| **エージェント名** |  削除 | エージェントリソースが削除される |
| **エージェント指示** |  削除 | エージェント定義の一部 |
| **エージェントツール** |  削除 | エージェント定義の一部 |
| **セッション ID** |  永続化 | 会話履歴が保持される |
| **セッションメッセージ** |  永続化 | データが保持される |
| **モデルデプロイメント** |  永続化 | 共有リソース |

### 自動削除の利点

#### 1. **コスト効率** 
```python
# 従来のアプローチ (エージェントが永続化)
agent_id = "agent-123"  # エージェントが Azure に残り、ガバナンスとメンテナンスの問題を引き起こす可能性がある

# Microsoft Foundry のアプローチ (エフェメラルエージェント)
async with Agent(...) as agent:
    # エージェントは必要な時だけ存在する
    await agent.run(query)
# エージェント削除
```

**利点:** 実行中のコンピュートにのみ料金を支払い、アイドル状態のエージェントストレージには支払わない

#### 2. **リソース管理** 🧹
```python
# 永続的エージェントの問題:
# - 古い/テストエージェントの蓄積
# - 名前空間の汚染
# - 手動クリーンアップが必要

# Microsoft Foundry の解決策:
async with Agent(name="TestAgent") as agent:
    # 作成とテスト
    pass
# 自動クリーンアップ - テストエージェントが残らない!
```

**利点:** "エージェントの氾濫"なし - Azure ワークスペースがクリーンに保たれる

#### 3. **セキュリティとコンプライアンス** 
```python
# PII アクセスを持つ機密エージェント
async with Agent(
    name="CustomerDataAgent",
    instructions="サポート目的で顧客の個人情報 (PII) にアクセスしてください",
) as agent:
    # エージェントは一時的に存在
    response = await agent.run("ユーザー情報を取得")
# エージェント定義が削除され、攻撃面が減少
```

**利点:** 短命な認証情報、セキュリティリスクの軽減

#### 4. **バージョン管理と GitOps** 
```python
# コード内のエージェント定義
async with Agent(
    name="V2TravelAgent",  # コード内のバージョン
    instructions="更新された指示..."  # ソース管理済み
) as agent:
    await agent.run(query)
```

**利点:** エージェントの動作はコードで定義され、Azure の状態ではない

### デメリットとトレードオフ

| デメリット | 軽減策 |
|--------------|------------|
| **実行ごとにエージェントが再作成される** | オーバーヘッドは最小限 (~100-300ms) |
| **ポータルでエージェントを表示できない** | 代わりにセッション/メッセージを表示 |
| **定義にコードが必要** | バージョン管理に適している |
| **長期実行エージェントなし** | 継続性にはセッションを使用 |

### 代替案: 永続的エージェント 

エージェントを永続化することもできます:

```python
agent = create_agent(name="MyAgent")  # エージェントがプラットフォームに残る
agent_id = agent.id  # "agent-abc123"

# 後で (別のセッション)
agent = get_agent(agent_id)  # 既存のエージェントを取得
response = agent.run("こんにちは")  # 永続的エージェントを使用
```

**永続的エージェントが意味を持つ場合:**
- 複雑で進化する指示を持つエージェント
- 複数のサービス間で共有されるエージェント
- 時間と共に学習/適応するエージェント
- 長期実行のバックグラウンドエージェント

**Microsoft Foundry の哲学:**
- エージェントは**コード** (リポジトリ内)
- セッションは**データ** (Azure 内)
- コンピュート (エフェメラル) と状態 (永続的) を分離

### Microsoft Foundry のベストプラクティス

#### 1. **エージェントをコードとして定義する**
```python
# 良い例: コード内のエージェント設定
def create_travel_agent(credential):
    return Agent(
        client=AzureAIAgentClient(credential=credential),
        name="TravelAgent",
        instructions="あなたは親切な旅行アシスタントです...",
        tools=[get_weather, get_attractions]
    )

# 使用方法
async with create_travel_agent(credential) as agent:
    response = await agent.run(query, session=session)
```

#### 2. **エージェント ID ではなくセッション ID を永続化する**
```python
# データベースに保存
session_data = {
    "user_id": user.id,
    "session_id": session.service_session_id,  #  これを保存
    # "agent_id": agent.id  #  これはしない (エージェントは削除される)
}

# 後で復元
async with create_travel_agent(credential) as agent:  # 新しいエージェント
    session = AgentSession(service_session_id=session_data["session_id"])  # 同じセッション
    response = await agent.run(query, session=session)
```

#### 3. **コンテキストマネージャーを使用する**
```python
#  良い例: 自動クリーンアップ
async with Agent(...) as agent:
    await agent.run(query)

#  避けるべき: 手動管理
agent = Agent(...)
await agent.run(query)
# エージェントが適切にクリーンアップされない可能性がある
```

### 比較表: Azure AI vs 従来型

| 側面 | Microsoft Foundry | 従来の永続型 |
|--------|------------------|----------------------|
| エージェントの寿命 | エフェメラル (秒単位) | 永続的 (日/月単位) |
| ストレージコスト | エージェントなし | エージェントの継続的コスト |
| クリーンアップ | 自動 | 手動が必要 |
| バージョン管理 | コード内 (Git) | プラットフォーム内 (API) |
| スケーリング | 無限のエージェント | クォータによる制限 |
| セッションの永続化 |  はい |  はい |
| エージェントの進化 | コード更新 | プラットフォーム更新 |
| マルチテナンシー | 実行ごとに分離 | 共有エージェント |

##  セッションライフサイクルの理解

### セッションの作成と管理

```python
# オプション 1: 自動でセッションを作成・管理させる
response = await agent.run(query)  # 毎回新しいセッション

# オプション 2: セッションを作成して再利用する (推奨)
session = agent.create_session()
response = await agent.run(query, session=session)  # 同じセッション

# オプション 3: 保存したセッション ID から再開する
session = AgentSession(service_session_id=saved_id)
response = await agent.run(query, session=session)
```

### Azure でのセッションストレージ

- セッションは **Microsoft Foundry に保存される**
- セッション ID はセッション間で**永続的**
- いつでも**会話を再開**できる
- Azure が**クリーンアップと保持**を管理する

### セッションを使用すべき場合

| ユースケース | セッション戦略 |
|----------|----------------|
| 一回限りの質問 | セッションなし (自動) |
| チャット会話 | 単一の永続的セッション |
| マルチユーザーアプリ | ユーザーごとに 1 つのセッション |
| セッションベース | セッションごとに新しいセッション |
| 長期アシスタント | セッション ID を保存・再開 |

## 🔬 実践例: エージェントライフサイクルデモ

エフェメラルエージェントの動作を実際に見てみましょう:

In [ ]:
async def agent_lifecycle_demo():
    """
    AzureOpenAIChatClient を使って、エージェントのセッション分離と会話再開を示します。
    """
    print("=== エージェント ライフサイクル デモ (AOAI) ===\n")

    if not (azure_openai_key and azure_openai_endpoint and api_version and azure_deployment):
        print("⚠️ Azure OpenAI 用の環境変数が不足しています。")
        print("   AZURE_OPENAI_API_KEY / AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_API_VERSION / AZURE_OPENAI_CHAT_DEPLOYMENT")
        return

    saved_session_state: dict | None = None

    print("--- セッション 1: エージェント作成→会話→state保存 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="EphemeralAgent_Session1",
            instructions="あなたは親切なアシスタントです。ユーザーから教えられた情報を覚えていてください。",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ エージェントを作成: {agent.name}")

        session = agent.create_session()
        query1 = "私のお気に入りの都市はバルセロナです。覚えておいてください。"
        print(f"ユーザー: {query1}")
        response1 = await agent.run(query1, session=session)
        print(f"エージェント: {response1.text}\n")

        saved_session_state = session.to_dict()
        print("✅ session state を保存")

    if not saved_session_state:
        print("⚠️ session state を保存できませんでした。")
        return

    print("--- セッション 2: 新しいエージェントで会話を再開 ---")
    async with (
        Agent(
            client=create_azure_openai_chat_client(),
            name="EphemeralAgent_Session2",
            instructions="あなたは親切なアシスタントです。ユーザーから教えられた情報を覚えていてください。",
            tools=[get_weather],
        ) as agent,
    ):
        print(f"✅ 新しいエージェントを作成: {agent.name}\n")

        session = AgentSession.from_dict(saved_session_state)

        query2 = "私のお気に入りの都市は何でしたか？"
        print(f"ユーザー: {query2}")
        response2 = await agent.run(query2, session=session)
        print(f"エージェント: {response2.text}\n")

        query3 = "その都市の天気はどうですか？"
        print(f"ユーザー: {query3}")
        response3 = await agent.run(query3, session=session)
        print(f"エージェント: {response3.text}\n")

    print("=" * 60)
    print("重要なインサイト:")
    print("=" * 60)
    print("1. エージェントはセッションごとに作成される")
    print("2. 会話継続には session state の保存/復元を使う")
    print("3. 異なるエージェントでも同じ会話を継続できる")
    print("=" * 60)

await agent_lifecycle_demo()

## リソースごとのスコープ/生存期間
上記の async with (...) は「気持ち悪い見た目」ですが、やってること自体は 複数の"後片付け付きリソース"を、入れ子で確実に生成→破棄する だけです。ポイントは スコープ（有効範囲） と 生存期間（いつ作られていつ閉じる/消えるか） が async with で厳密に決まることです。

この1ブロックの中には、少なくとも次の "寿命の違うもの" が混ざっています。

- `credential`（認証ハンドル）
  - 生存期間: `async with DefaultAzureCredential()` のブロック内だけ有効
  - 終了時: `credential` が close され、内部の HTTP 接続なども閉じられることがある

- `create_azure_openai_chat_client()` が返すクライアント（`AzureOpenAIChatClient`）
  - 生存期間: ここでは `Agent(... ) as agent` の中で使われるだけ（その場限り）
  - 終了時: `Agent` の終了に合わせてクライアント利用も終了する

- `agent`（Foundry 側に作られる「短命エージェント」）
  - 生存期間: `async with ... as agent:` のブロック内だけ
  - 終了時: ブロックを抜けた瞬間に **サービス側のエージェント定義が削除される**（= エフェメラル）

- `session`（会話セッション）
  - 生存期間: 2種類あり得る
    - **サービス管理セッション**なら `session.service_session_id` が払い出され、IDを保存して後で復元できる（エージェントが消えてもセッションは残る）
    - そうでないならローカル保持のみで、プロセス終了や state を保存しない限り続かない

つまり「エージェントは短命、セッションは永続（になることがある）」という設計を、この `async with` が強制的に実現しています。

## "気持ち悪さ" の正体（なぜ読みにくいか）
- 1行の `async with` に **3階層分のライフサイクル**（credential / client / agent）を畳み込んでいる
- さらに `.as_agent(...)` が「クライアントから"エージェント用のコンテキストマネージャ"を生成する」という二段構えになっていて、生成と破棄の境界が見えづらい

##  重要なポイント

### セッションの利点
1. **コンテキスト認識** - エージェントが以前のやり取りを記憶
2. **自然な会話** - ユーザーが情報を繰り返す必要がない
3. **より良い UX** - 人間と話しているような感覚
4. **ステートフルなインタラクション** - 以前の応答を基に構築

### ベストプラクティス
1. **会話にはセッションを使用** - マルチターン対話には常に使用する
2. **セッション ID を保存** - ユーザーのデータベースに保存
3. **会話ごとに 1 つのセッション** - 異なるトピックを混在させない
4. **古いセッションをクリーンアップ** - 会話終了時に削除 (Azure がこれを支援)

### 一般的なパターン

**Web チャットアプリケーション:**
```python
# ユーザーセッションに session_id を保存
if not session_store.get('session_id'):
    session = agent.create_session()
    session_store['session_id'] = session.service_session_id
else:
    session = AgentSession(service_session_id=session_store['session_id'])
```

**カスタマーサポート:**
```python
# サポートチケットごとに 1 つのセッション
session_id = ticket.get_session_id()
session = AgentSession(service_session_id=session_id)
```

##  練習問題

1. **予約会話の作成** - 旅行の最初から予約までの完全な計画を立てる
2. **目的地の比較** - エージェントに複数の都市を記憶して比較させる
3. **嗜好プロファイルの構築** - 会話を通じてエージェントがユーザーの好みを学習する
4. **セッション管理の実装** - 辞書からセッション ID を保存/読み込む

In [ ]:
# Exercise: Create a conversation where the agent helps compare 3 destinations
# and remembers all the details discussed about each one

async def compare_destinations_exercise():
    """
    Your turn! Create a conversation that:
    1. Discusses Paris (weather, attractions, budget)
    2. Discusses Tokyo (same info)
    3. Discusses Barcelona (same info)
    4. Asks agent to compare all three and recommend
    
    The agent should remember ALL details from the conversation!
    """
    # Your code here
    pass

# Uncomment to test your solution
# await compare_destinations_exercise()

##  次のステップ

素晴らしい! ステートフルでコンテキスト認識する会話の作成方法を理解しました。

しかし、エージェントにはまだ制限があります:
-  セッション間でユーザーの好みを記憶しない
-  ユーザーに関する長期的な知識を保存できない
-  セッションの会話履歴が非常に長くなる可能性がある

**チュートリアル 03: コンテキストとメモリ**では、以下を学習します:
- エージェントに永続的なメモリを追加する
- ユーザーの好みとプロファイルを保存する
- よりスマートなエージェントのためのコンテキストプロバイダーを使用する
- 会話コンテキストを効率的に管理する

---

### クイックリファレンス

**永続的セッションの作成:**
```python
session = agent.create_session()
response = await agent.run(query, session=session)
```

**保存と再開:**
```python
# 保存
session_id = session.service_session_id

# 再開
session = AgentSession(service_session_id=session_id)
```

**複数の会話:**
```python
alice_session = agent.create_session()
bob_session = agent.create_session()
```